# 09. GRRF-inspired parametrų paieška tik TRAIN GroupKFold

Šis etapas **nevertina galutinio TEST**. Jokio TEST Macro-F1, Balanced Accuracy ar painiavos matricos.

Tikslas: nested-protokolu (feature selection **kiekviename fold'e** ant fold TRAIN) palyginti GRRF-inspired + `LinearSVC(C=0.01)` su 561 požymių SVM, ir išrinkti vieną parametrų kombinaciją pagal **mean CV Macro-F1**.

08 notebook ir jo TEST rezultatai neperrašomi.

## Kokie parametrai matematiškai keičia atranką

Straipsnio (8) lygtis: `α_k = (1−γ)λ + γ (Impscore_k / Impscore_max)`.

- **γ=1** (Thakur ir Biswas: maksimali bauda): `α_k = Imp_k/Imp_max`. **λ neturi įtakos.** Fiktyvaus λ tinklelio prie γ=1 nėra.
- **γ=0.5** (Deng GRRF pavyzdys): λ **turi** įtaką, nes `(1−γ)λ = 0.5λ`.
- **n_tree**: kiek regularizuotų medžių; `feaSet` yra sąjunga, tad daugiau medžių paprastai daugiau požymių.
- **mtry**: kiek kandidatų mazge (straipsnio RF preliminaries: klasifikacijai `√p`).

Nedidelis tinklelis (5 GRRF kombinacijos + 1 baseline):

| id | γ | λ | n_tree | mtry |
|---|---|---|---|---|
| G1 | 1.0 | nenaudojamas | 50 | √p |
| G2 | 1.0 | nenaudojamas | 100 | √p |
| G3 | 1.0 | nenaudojamas | 100 | 2√p |
| G4 | 0.5 | 1.0 | 100 | √p |
| G5 | 0.5 | 0.1 | 100 | √p |

G2 atitinka 08 etapo fiksuotus parametrus. G4 vs G5 izoliuoja λ tik ten, kur (8) lygtis jį naudoja.

**Atrankos taisyklė:** didžiausias mean CV Macro-F1. Jei kitos kombinacijos atsilieka ≤ 0.003, tarp jų imama mažesnė vidutinė požymių skaičiaus reikšmė. Jei ir tai lygų — mažesnis CV Macro-F1 std.

In [1]:
from pathlib import Path
import time
import warnings

import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import balanced_accuracy_score, f1_score
from sklearn.model_selection import GroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC

RANDOM_STATE = 42
N_SPLITS = 5
SIMILAR_F1 = 0.003
np.random.seed(RANDOM_STATE)
print(f"RANDOM_STATE = {RANDOM_STATE}")
print("Galutinis TEST šiame notebook NENAUDOJAMAS metrikoms.")

RANDOM_STATE = 42
Galutinis TEST šiame notebook NENAUDOJAMAS metrikoms.


In [2]:
cwd = Path.cwd().resolve()
project_root = None
for cand in [cwd, cwd.parent]:
    if (cand / "results").exists() and (cand / "notebooks").exists():
        project_root = cand
        break
if project_root is None:
    project_root = cwd.parent if cwd.name == "notebooks" else cwd

results_dir = project_root / "results"
split_path = results_dir / "subject_disjoint_split.npz"
split = np.load(split_path, allow_pickle=True)

X_train = np.asarray(split["X_train_final"], dtype=np.float64)
y_train = np.asarray(split["y_train_final"], dtype=np.int64)
subjects_train = np.asarray(split["subjects_train_final"], dtype=np.int64)
subjects_test = np.asarray(split["subjects_test_final"], dtype=np.int64)

train_ids = sorted(set(subjects_train.tolist()))
test_ids = sorted(set(subjects_test.tolist()))
overlap = sorted(set(train_ids) & set(test_ids))

print(f"Skaidymas: {split_path}")
print(f"X_train: {X_train.shape} (TEST masyvai metrikoms neimami)")
print(f"TRAIN subject ID ({len(train_ids)}): {train_ids}")
print(f"TEST subject ID tik sankirtai ({len(test_ids)}): {test_ids}")
print(f"TRAIN ∩ TEST: {overlap}")
assert overlap == []
assert X_train.shape == (7144, 561)
assert len(train_ids) == 21
print("Assertion: sankirta tuščia. X_test/y_test į atmintį kėlimo metrikoms nėra.")

TEST_USED_FOR_METRICS = False

Skaidymas: C:\Users\Lenovo\Desktop\Studijos 2026\Magistras\Intelektualios sistemos\Egzaminas\EGZ_Dovaldas_Dovidovic_EKSFM-26\results\subject_disjoint_split.npz
X_train: (7144, 561) (TEST masyvai metrikoms neimami)
TRAIN subject ID (21): [1, 2, 3, 4, 5, 6, 7, 8, 11, 12, 14, 15, 17, 19, 20, 21, 22, 23, 26, 27, 30]
TEST subject ID tik sankirtai (9): [9, 10, 13, 16, 18, 24, 25, 28, 29]
TRAIN ∩ TEST: []
Assertion: sankirta tuščia. X_test/y_test į atmintį kėlimo metrikoms nėra.


In [3]:
def gini_parent(counts):
    n = counts.sum()
    if n <= 0:
        return 0.0
    return 1.0 - float(np.dot(counts, counts)) / float(n * n)


def best_threshold_and_gain(x_col, y_node, n_classes, min_leaf):
    n = x_col.shape[0]
    if n < 2 * min_leaf:
        return -1.0, np.nan
    order = np.argsort(x_col, kind="mergesort")
    x_s = x_col[order]
    y_s = y_node[order]
    if x_s[0] == x_s[-1]:
        return -1.0, np.nan
    oh = np.zeros((n, n_classes), dtype=np.float64)
    oh[np.arange(n), y_s] = 1.0
    left = np.cumsum(oh, axis=0)
    total = left[-1]
    g_parent = gini_parent(total)
    i = np.arange(min_leaf, n - min_leaf + 1)
    valid = x_s[i] != x_s[i - 1]
    if not np.any(valid):
        return -1.0, np.nan
    left_c = left[i - 1]
    left_n = i.astype(np.float64)
    right_c = total - left_c
    right_n = float(n) - left_n
    g_left = 1.0 - np.sum(left_c * left_c, axis=1) / (left_n * left_n)
    g_right = 1.0 - np.sum(right_c * right_c, axis=1) / (right_n * right_n)
    gain = g_parent - (left_n / float(n)) * g_left - (right_n / float(n)) * g_right
    gain = np.where(valid, gain, -np.inf)
    j = int(np.argmax(gain))
    thr = 0.5 * (x_s[i[j]] + x_s[i[j] - 1])
    return float(gain[j]), float(thr)


def grow_regularized_tree(X, y, rng, alpha, max_features, min_samples_split, min_samples_leaf):
    n_samples, n_features = X.shape
    n_classes = int(y.max()) + 1
    used = set()
    used_in_tree = set()

    def majority(idx):
        return int(np.argmax(np.bincount(y[idx], minlength=n_classes)))

    def recurse(idx):
        y_n = y[idx]
        if idx.size < min_samples_split or np.unique(y_n).size == 1:
            return {"leaf": True, "pred": majority(idx)}
        mtry = min(max_features, n_features)
        cand = rng.choice(n_features, size=mtry, replace=False)
        best_gain, best_k, best_thr = -1.0, None, None
        x_node = X[idx]
        for k in cand:
            raw_gain, thr = best_threshold_and_gain(x_node[:, k], y_n, n_classes, min_samples_leaf)
            if raw_gain < 0 or not np.isfinite(thr):
                continue
            # (6)–(7): k ∈ F — be baudos; k ∉ F — × α_k iš (8)
            gain = raw_gain if k in used else raw_gain * float(alpha[k])
            if gain > best_gain:
                best_gain, best_k, best_thr = gain, int(k), thr
        if best_k is None or best_gain <= 0:
            return {"leaf": True, "pred": majority(idx)}
        used.add(best_k)
        used_in_tree.add(best_k)
        left_mask = x_node[:, best_k] <= best_thr
        left_idx, right_idx = idx[left_mask], idx[~left_mask]
        if left_idx.size == 0 or right_idx.size == 0:
            return {"leaf": True, "pred": majority(idx)}
        return {
            "leaf": False,
            "feature": best_k,
            "threshold": best_thr,
            "left": recurse(left_idx),
            "right": recurse(right_idx),
        }

    recurse(np.arange(n_samples, dtype=np.int64))
    return used_in_tree


def resolve_mtry(n_features, spec):
    if spec == "sqrt":
        return max(1, int(np.sqrt(n_features)))
    if spec == "2sqrt":
        return max(1, min(n_features, int(2 * np.sqrt(n_features))))
    raise ValueError(spec)


def grrf_inspired_select_fold(X_fit, y_fit, gamma, lambda_penalty, n_estimators, mtry_spec, random_state):
    """Atranka tik fold TRAIN. α_k pagal (8)."""
    n_features = X_fit.shape[1]
    mtry = resolve_mtry(n_features, mtry_spec)
    rf = RandomForestClassifier(
        n_estimators=n_estimators,
        max_features="sqrt",
        random_state=random_state,
        n_jobs=-1,
    )
    rf.fit(X_fit, y_fit)
    imp = rf.feature_importances_.astype(np.float64)
    imp_max = float(imp.max())
    if imp_max <= 0:
        raise RuntimeError("RF importance visur 0")
    imp_norm = imp / imp_max
    # (8) α_k = (1-γ)λ + γ Imp_norm; jei γ=1, λ nedaugina nieko
    alpha = (1.0 - gamma) * lambda_penalty + gamma * imp_norm
    alpha = np.clip(alpha, 0.0, 1.0)
    _, y_enc = np.unique(y_fit, return_inverse=True)
    rng = np.random.RandomState(random_state)
    selected = set()
    for _ in range(n_estimators):
        boot = rng.randint(0, X_fit.shape[0], size=X_fit.shape[0])
        selected |= grow_regularized_tree(
            X_fit[boot],
            y_enc[boot],
            rng,
            alpha,
            max_features=mtry,
            min_samples_split=2,
            min_samples_leaf=1,
        )
    return np.array(sorted(selected), dtype=np.int64)


def make_svm():
    return Pipeline(
        [
            ("scaler", StandardScaler()),
            (
                "clf",
                LinearSVC(
                    C=0.01,
                    random_state=RANDOM_STATE,
                    max_iter=10000,
                    dual="auto",
                ),
            ),
        ]
    )

## Nested GroupKFold tik TRAIN

Kiekviename fold'e: importance, GRRF-inspired mask, scaler ir SVM mokomi **tik** fold TRAIN. VALIDATION — nematyti TRAIN žmonės. Feature mask **ne** skaičiuojamas ant viso TRAIN prieš CV.

In [4]:
cv = GroupKFold(n_splits=N_SPLITS)
fold_groups = []
for fold_i, (tr, va) in enumerate(cv.split(X_train, y_train, groups=subjects_train), start=1):
    g_tr = sorted(set(subjects_train[tr].tolist()))
    g_va = sorted(set(subjects_train[va].tolist()))
    inter = sorted(set(g_tr) & set(g_va))
    fold_groups.append((g_tr, g_va, inter))
    print(
        f"Fold {fold_i}: TRAIN people {len(g_tr)} {g_tr} | VAL people {len(g_va)} {g_va} | ∩ {inter}"
    )
    assert inter == [], f"Fold {fold_i} subject leakage {inter}"
    assert set(g_va).isdisjoint(set(test_ids)), "VAL subject pateko į galutinio TEST ID — neturėtų būti, bet TEST vis tiek nevertinamas"
print("Visi fold'ai: VAL žmonės nesusikerta su fold TRAIN.")

GRID = [
    {
        "id": "G1",
        "gamma": 1.0,
        "lambda_penalty": np.nan,
        "lambda_used": False,
        "n_estimators": 50,
        "mtry": "sqrt",
        "note": "γ=1 Thakur; λ nenaudojamas; n_tree=50; mtry=√p",
    },
    {
        "id": "G2",
        "gamma": 1.0,
        "lambda_penalty": np.nan,
        "lambda_used": False,
        "n_estimators": 100,
        "mtry": "sqrt",
        "note": "γ=1; n_tree=100; mtry=√p (08 etapo nustatymai)",
    },
    {
        "id": "G3",
        "gamma": 1.0,
        "lambda_penalty": np.nan,
        "lambda_used": False,
        "n_estimators": 100,
        "mtry": "2sqrt",
        "note": "γ=1; n_tree=100; mtry=2√p",
    },
    {
        "id": "G4",
        "gamma": 0.5,
        "lambda_penalty": 1.0,
        "lambda_used": True,
        "n_estimators": 100,
        "mtry": "sqrt",
        "note": "γ=0.5 Deng; λ=1.0 turi įtaką; n_tree=100; mtry=√p",
    },
    {
        "id": "G5",
        "gamma": 0.5,
        "lambda_penalty": 0.1,
        "lambda_used": True,
        "n_estimators": 100,
        "mtry": "sqrt",
        "note": "γ=0.5; λ=0.1 turi įtaką; n_tree=100; mtry=√p",
    },
]
print("GRRF tinklelis:", [g["id"] for g in GRID])
print("λ derinamas TIK G4 ir G5 (γ=0.5). G1–G3 λ netuninginamas.")

Fold 1: TRAIN people 17 [1, 4, 5, 6, 7, 8, 11, 14, 15, 17, 19, 20, 22, 23, 26, 27, 30] | VAL people 4 [2, 3, 12, 21] | ∩ []
Fold 2: TRAIN people 17 [2, 3, 4, 5, 6, 7, 8, 12, 15, 17, 19, 20, 21, 22, 23, 27, 30] | VAL people 4 [1, 11, 14, 26] | ∩ []
Fold 3: TRAIN people 17 [1, 2, 3, 4, 5, 8, 11, 12, 14, 15, 17, 19, 21, 22, 23, 26, 27] | VAL people 4 [6, 7, 20, 30] | ∩ []
Fold 4: TRAIN people 16 [1, 2, 3, 4, 6, 7, 11, 12, 14, 17, 20, 21, 22, 23, 26, 30] | VAL people 5 [5, 8, 15, 19, 27] | ∩ []
Fold 5: TRAIN people 17 [1, 2, 3, 5, 6, 7, 8, 11, 12, 14, 15, 19, 20, 21, 26, 27, 30] | VAL people 4 [4, 17, 22, 23] | ∩ []
Visi fold'ai: VAL žmonės nesusikerta su fold TRAIN.
GRRF tinklelis: ['G1', 'G2', 'G3', 'G4', 'G5']
λ derinamas TIK G4 ir G5 (γ=0.5). G1–G3 λ netuninginamas.


In [5]:
def cv_baseline_561():
    f1s, bas, nfeats = [], [], []
    for fold_i, (tr, va) in enumerate(cv.split(X_train, y_train, groups=subjects_train), start=1):
        pipe = make_svm()
        pipe.fit(X_train[tr], y_train[tr])
        pred = pipe.predict(X_train[va])
        f1s.append(float(f1_score(y_train[va], pred, average="macro")))
        bas.append(float(balanced_accuracy_score(y_train[va], pred)))
        nfeats.append(561)
        print(f"  BASE fold {fold_i}: Macro-F1={f1s[-1]:.6f} BA={bas[-1]:.6f} n=561 VAL_n={len(va)}")
    return {
        "id": "BASE",
        "gamma": np.nan,
        "lambda_penalty": np.nan,
        "lambda_used": False,
        "n_estimators": np.nan,
        "mtry": "all",
        "note": "LinearSVC visiems 561, be GRRF",
        "mean_cv_macro_f1": float(np.mean(f1s)),
        "std_cv_macro_f1": float(np.std(f1s, ddof=1)),
        "mean_cv_ba": float(np.mean(bas)),
        "std_cv_ba": float(np.std(bas, ddof=1)),
        "mean_n_features": float(np.mean(nfeats)),
        "std_n_features": 0.0,
        "fold_macro_f1": f1s,
        "fold_ba": bas,
        "fold_n_features": nfeats,
    }


def cv_grrf(cfg):
    lam = 1.0 if not cfg["lambda_used"] else float(cfg["lambda_penalty"])
    f1s, bas, nfeats = [], [], []
    t0 = time.perf_counter()
    for fold_i, (tr, va) in enumerate(cv.split(X_train, y_train, groups=subjects_train), start=1):
        sel = grrf_inspired_select_fold(
            X_train[tr],
            y_train[tr],
            gamma=float(cfg["gamma"]),
            lambda_penalty=lam,
            n_estimators=int(cfg["n_estimators"]),
            mtry_spec=cfg["mtry"],
            random_state=RANDOM_STATE + fold_i,
        )
        if sel.size == 0:
            raise RuntimeError(f"{cfg['id']} fold {fold_i}: atrinkta 0 požymių")
        pipe = make_svm()
        pipe.fit(X_train[tr][:, sel], y_train[tr])
        pred = pipe.predict(X_train[va][:, sel])
        f1s.append(float(f1_score(y_train[va], pred, average="macro")))
        bas.append(float(balanced_accuracy_score(y_train[va], pred)))
        nfeats.append(int(sel.size))
        print(
            f"  {cfg['id']} fold {fold_i}: n_feat={sel.size} "
            f"Macro-F1={f1s[-1]:.6f} BA={bas[-1]:.6f} ({time.perf_counter() - t0:.1f}s)"
        )
    return {
        **{k: cfg[k] for k in ["id", "gamma", "lambda_penalty", "lambda_used", "n_estimators", "mtry", "note"]},
        "mean_cv_macro_f1": float(np.mean(f1s)),
        "std_cv_macro_f1": float(np.std(f1s, ddof=1)),
        "mean_cv_ba": float(np.mean(bas)),
        "std_cv_ba": float(np.std(bas, ddof=1)),
        "mean_n_features": float(np.mean(nfeats)),
        "std_n_features": float(np.std(nfeats, ddof=1)),
        "fold_macro_f1": f1s,
        "fold_ba": bas,
        "fold_n_features": nfeats,
    }


print("=== BASELINE 561 LinearSVC, GroupKFold TRAIN ===")
rows = [cv_baseline_561()]
for cfg in GRID:
    print(f"=== {cfg['id']}: {cfg['note']} ===")
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        rows.append(cv_grrf(cfg))

assert TEST_USED_FOR_METRICS is False
print("TEST metrikos vis dar neskaičiuotos.")

=== BASELINE 561 LinearSVC, GroupKFold TRAIN ===


  BASE fold 1: Macro-F1=0.977266 BA=0.976138 n=561 VAL_n=1371


  BASE fold 2: Macro-F1=0.939883 BA=0.941894 n=561 VAL_n=1378


  BASE fold 3: Macro-F1=0.949581 BA=0.950332 n=561 VAL_n=1370


  BASE fold 4: Macro-F1=0.957760 BA=0.958298 n=561 VAL_n=1647


...

  G5 fold 1: n_feat=521 Macro-F1=0.977266 BA=0.976138 (49.8s)


  G5 fold 2: n_feat=502 Macro-F1=0.939883 BA=0.941888 (94.7s)


  G5 fold 3: n_feat=495 Macro-F1=0.952015 BA=0.952548 (137.5s)


  G5 fold 4: n_feat=517 Macro-F1=0.957200 BA=0.957940 (178.8s)


  G5 fold 5: n_feat=519 Macro-F1=0.982312 BA=0.981984 (223.1s)
TEST metrikos vis dar neskaičiuotos.


In [6]:
summary = pd.DataFrame(
    [
        {
            "id": r["id"],
            "gamma": r["gamma"],
            "lambda_penalty": r["lambda_penalty"],
            "lambda_used": r["lambda_used"],
            "n_estimators": r["n_estimators"],
            "mtry": r["mtry"],
            "mean_cv_macro_f1": r["mean_cv_macro_f1"],
            "std_cv_macro_f1": r["std_cv_macro_f1"],
            "mean_cv_ba": r["mean_cv_ba"],
            "std_cv_ba": r["std_cv_ba"],
            "mean_n_features": r["mean_n_features"],
            "std_n_features": r["std_n_features"],
            "fold_macro_f1": ";".join(f"{x:.8f}" for x in r["fold_macro_f1"]),
            "fold_ba": ";".join(f"{x:.8f}" for x in r["fold_ba"]),
            "fold_n_features": ";".join(str(x) for x in r["fold_n_features"]),
            "note": r["note"],
        }
        for r in rows
    ]
)

grrf_only = summary[summary["id"] != "BASE"].copy()
best_f1 = float(grrf_only["mean_cv_macro_f1"].max())
similar = grrf_only[grrf_only["mean_cv_macro_f1"] >= best_f1 - SIMILAR_F1].copy()
similar = similar.sort_values(
    ["mean_n_features", "std_cv_macro_f1", "mean_cv_macro_f1"],
    ascending=[True, True, False],
)
chosen_id = str(similar.iloc[0]["id"])
chosen = summary.loc[summary["id"] == chosen_id].iloc[0]
base = summary.loc[summary["id"] == "BASE"].iloc[0]

print("Visos kombinacijos (TRAIN GroupKFold):")
print(
    summary[
        [
            "id",
            "gamma",
            "lambda_penalty",
            "n_estimators",
            "mtry",
            "mean_cv_macro_f1",
            "std_cv_macro_f1",
            "mean_cv_ba",
            "mean_n_features",
            "std_n_features",
        ]
    ].to_string(index=False)
)
print()
print(f"Panašumo slenkstis: geriausias F1 − {SIMILAR_F1}")
print("Kandidatai (F1 ≥ best−0.003):")
print(similar[["id", "mean_cv_macro_f1", "mean_n_features", "std_cv_macro_f1"]].to_string(index=False))
print()
print("Pasirinkta:", chosen_id)
print(chosen[["gamma", "lambda_penalty", "lambda_used", "n_estimators", "mtry",
              "mean_cv_macro_f1", "std_cv_macro_f1", "mean_cv_ba", "mean_n_features"]].to_string())
print()
print("Baseline mean CV Macro-F1:", float(base["mean_cv_macro_f1"]))
print("Pasirinkta vs BASE Δ F1:", float(chosen["mean_cv_macro_f1"]) - float(base["mean_cv_macro_f1"]))
print("Ar viršija / siekia BASE (Δ≥0):", float(chosen["mean_cv_macro_f1"]) >= float(base["mean_cv_macro_f1"]))

out_path = results_dir / "grrf_inspired_groupkfold_tuning.csv"
summary.to_csv(out_path, index=False)
print("Išsaugota:", out_path)
print("08 failai neperrašyti. TEST nevertintas: TEST_USED_FOR_METRICS =", TEST_USED_FOR_METRICS)

Visos kombinacijos (TRAIN GroupKFold):
  id  gamma  lambda_penalty  n_estimators  mtry  mean_cv_macro_f1  std_cv_macro_f1  mean_cv_ba  mean_n_features  std_n_features
BASE    NaN             NaN           NaN   all          0.961494         0.018258    0.961877            561.0        0.000000
  G1    1.0             NaN          50.0  sqrt          0.949908         0.017987    0.950353            212.2        9.148770
  G2    1.0             NaN         100.0  sqrt          0.955026         0.016008    0.955533            243.4        4.505552
  G3    1.0             NaN         100.0 2sqrt          0.947490         0.013083    0.948003            175.4        7.797435
  G4    0.5             1.0         100.0  sqrt          0.961383         0.018288    0.961772            560.4        0.547723
  G5    0.5             0.1         100.0  sqrt          0.961735         0.017728    0.962100            510.8       11.584472

Panašumo slenkstis: geriausias F1 − 0.003
Kandidatai (F1 ≥ best−

Išsaugota: C:\Users\Lenovo\Desktop\Studijos 2026\Magistras\Intelektualios sistemos\Egzaminas\EGZ_Dovaldas_Dovidovic_EKSFM-26\results\grrf_inspired_groupkfold_tuning.csv
08 failai neperrašyti. TEST nevertintas: TEST_USED_FOR_METRICS = False


## Data leakage auditas (šis etapas)

Galutinis TEST nenaudotas `fit`, `predict` ir metrikoms. Žemiau — patikros iš vykdyto kodo.

In [7]:
print("=== DATA LEAKAGE AUDITAS ===")
print("1) Subject sankirta TRAIN/TEST:", overlap, "tuščia=", overlap == [])
print("2) CV groups = subjects_train, n_splits=", N_SPLITS)
for i, (g_tr, g_va, inter) in enumerate(fold_groups, start=1):
    print(f"   fold {i} TRAIN∩VAL={inter} VAL∩galutinisTEST={sorted(set(g_va) & set(test_ids))}")
print("3) Feature selection: grrf_inspired_select_fold(X_train[tr], y_train[tr]) kiekviename fold.")
print("4) Scaler+SVM: Pipeline.fit tik fold TRAIN stulpeliams; predict tik fold VAL.")
print("5) LinearSVC C=0.01 fiksuotas, neieškomas TEST.")
print("6) TEST_USED_FOR_METRICS =", TEST_USED_FOR_METRICS)
print("7) X_test/y_test šiame notebook nebuvo užkrauti (tik TEST subject ID sankirtai).")
print("8) Geriausia kombinacija išrinkta iš CSV mean_cv_macro_f1, ne iš TEST.")
print("Pasirinktas kitas etapas (vieną kartą TEST):", chosen_id,
      "γ=", chosen["gamma"], "λ_used=", chosen["lambda_used"],
      "λ=", chosen["lambda_penalty"], "n_tree=", chosen["n_estimators"],
      "mtry=", chosen["mtry"])

=== DATA LEAKAGE AUDITAS ===
1) Subject sankirta TRAIN/TEST: [] tuščia= True
2) CV groups = subjects_train, n_splits= 5
   fold 1 TRAIN∩VAL=[] VAL∩galutinisTEST=[]
   fold 2 TRAIN∩VAL=[] VAL∩galutinisTEST=[]
   fold 3 TRAIN∩VAL=[] VAL∩galutinisTEST=[]
   fold 4 TRAIN∩VAL=[] VAL∩galutinisTEST=[]
   fold 5 TRAIN∩VAL=[] VAL∩galutinisTEST=[]
3) Feature selection: grrf_inspired_select_fold(X_train[tr], y_train[tr]) kiekviename fold.
4) Scaler+SVM: Pipeline.fit tik fold TRAIN stulpeliams; predict tik fold VAL.
5) LinearSVC C=0.01 fiksuotas, neieškomas TEST.
6) TEST_USED_FOR_METRICS = False
7) X_test/y_test šiame notebook nebuvo užkrauti (tik TEST subject ID sankirtai).
8) Geriausia kombinacija išrinkta iš CSV mean_cv_macro_f1, ne iš TEST.
Pasirinktas kitas etapas (vieną kartą TEST): G5 γ= 0.5 λ_used= True λ= 0.1 n_tree= 100.0 mtry= sqrt
